# Module 12 lab — Prefill versus incremental decode

This notebook is self-contained. It measures prefill, one-token decode, prefill/decode curves, and end-to-end generation on one NVIDIA T4.

Cache construction is kept outside the timed decode region.

## Lab A — Setup

Run this once in a Lightning AI notebook or terminal if the dependencies are not already installed.

In [ ]:
# Install the versions used by this lab. Restart the kernel only if the runtime asks for it.
%pip install -U "transformers>=4.45,<5" accelerate safetensors pandas matplotlib

In [ ]:
# Import dependencies and select the first CUDA device.
import gc
import statistics
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), 'Select a T4 GPU runtime.'
device = torch.device('cuda:0')
model_id = 'Qwen/Qwen2.5-0.5B-Instruct'

print('GPU:', torch.cuda.get_device_name(0))
print('CUDA capability:', torch.cuda.get_device_capability(0))
print('PyTorch:', torch.__version__)

In [ ]:
# Load the tokenizer and FP16 model once; every experiment reuses these objects.
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
).to(device).eval()

print('Layers:', model.config.num_hidden_layers)
print('KV heads:', model.config.num_key_value_heads)
print('Model dtype:', next(model.parameters()).dtype)
print('Parameters:', f'{sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# CUDA is asynchronous, so use CUDA events to time GPU execution rather than CPU dispatch.
def gpu_time_ms(fn):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    result = fn()
    end.record()
    end.synchronize()
    return start.elapsed_time(end), result

## Experiment 1 — Prefill latency versus prompt length

If prompt length doubles, prefill time does not have to double: projection work scales roughly linearly, attention has a quadratic component, and GEMM efficiency changes with shape.

In [ ]:
# Measure a full prompt forward pass and return the median GPU time.
@torch.inference_mode()
def measure_prefill(seq_len, repeats=8):
    input_ids = torch.randint(
        100, 50000, (1, seq_len), device=device
    )

    def run():
        return model(
            input_ids=input_ids,
            use_cache=True,
            return_dict=True,
        )

    # Warm up model kernels and allocator behavior before collecting samples.
    for _ in range(2):
        output = run()
        del output
    torch.cuda.synchronize()

    times = []
    for _ in range(repeats):
        milliseconds, output = gpu_time_ms(run)
        times.append(milliseconds)
        del output
    del input_ids
    return statistics.median(times)

prefill_rows = []
for seq_len in [64, 128, 256, 512, 1024]:
    milliseconds = measure_prefill(seq_len)
    prefill_rows.append({
        'prompt_tokens': seq_len,
        'prefill_ms': milliseconds,
        'prompt_tokens_per_sec': seq_len / (milliseconds / 1000),
    })

prefill_df = pd.DataFrame(prefill_rows)
display(prefill_df)

In [ ]:
# Plot both latency and throughput; the two curves answer different questions.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(prefill_df['prompt_tokens'], prefill_df['prefill_ms'], marker='o')
axes[0].set_xlabel('Prompt length (tokens)')
axes[0].set_ylabel('Prefill GPU time (ms)')
axes[0].set_title('T4 — Prefill latency')
axes[1].plot(prefill_df['prompt_tokens'], prefill_df['prompt_tokens_per_sec'], marker='o')
axes[1].set_xlabel('Prompt length (tokens)')
axes[1].set_ylabel('Prompt tokens/sec')
axes[1].set_title('T4 — Prefill throughput')
for axis in axes:
    axis.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Standard Transformers may compute logits for every prompt position. That makes this a useful baseline for Hugging Face execution, but not a direct measurement of an optimized vLLM prefill path.

## Experiment 2 — Decode latency versus context length

Decode processes one new token, but that query attends to the previously cached K/V history. The cache is built before timing.

In [ ]:
# Build a fresh cache for every trial so cache mutation cannot change the next sample.
@torch.inference_mode()
def measure_decode(context_len, repeats=8):
    prompt = torch.randint(
        100, 50000, (1, context_len), device=device
    )
    times = []
    cache_lengths = []

    # Two warmup trials are discarded; each trial creates its cache outside timing.
    for trial in range(repeats + 2):
        prefill = model(
            input_ids=prompt,
            use_cache=True,
            return_dict=True,
        )
        cache = prefill.past_key_values
        next_token = prefill.logits[:, -1:, :].argmax(dim=-1)
        cache_before = cache.get_seq_length()

        # Only this one-token forward pass belongs to decode timing.
        milliseconds, result = gpu_time_ms(
            lambda: model(
                input_ids=next_token,
                past_key_values=cache,
                use_cache=True,
                return_dict=True,
            )
        )
        cache_after = result.past_key_values.get_seq_length()
        if trial >= 2:
            times.append(milliseconds)
            cache_lengths.append((cache_before, cache_after))
        del prefill, cache, next_token, result

    del prompt
    gc.collect()
    torch.cuda.empty_cache()
    cache_before, cache_after = cache_lengths[-1]
    return statistics.median(times), cache_before, cache_after

decode_rows = []
for context_len in [64, 256, 512, 1024]:
    milliseconds, cache_before, cache_after = measure_decode(context_len)
    decode_rows.append({
        'context_tokens': context_len,
        'decode_ms': milliseconds,
        'cache_length_before': cache_before,
        'cache_length_after': cache_after,
    })

decode_df = pd.DataFrame(decode_rows)
display(decode_df)
assert (decode_df['cache_length_after'] == decode_df['cache_length_before'] + 1).all()

In [ ]:
# Plot incremental decode latency against the cached context length.
plt.figure(figsize=(8, 4))
plt.plot(decode_df['context_tokens'], decode_df['decode_ms'], marker='o')
plt.xlabel('Previously cached tokens')
plt.ylabel('Incremental forward time (ms)')
plt.title('T4 — Decode latency versus context length')
plt.grid(alpha=0.3)
plt.show()

For this small Qwen model, context-length effects may be modest or noisy because weight projections, launch overhead, allocations, kernel selection, and GQA cache behavior also contribute. The important check is that the new token increases the cache length by one.

## Experiment 3 — Compare prefill and decode

The operations do different amounts of useful work, so equal milliseconds do not mean equal efficiency. Compare latency together with tokens/sec, arithmetic work, memory traffic, TTFT, and ITL.

In [ ]:
# Join the measurements where prompt and cached-context lengths overlap.
merged = pd.merge(
    prefill_df,
    decode_df,
    left_on='prompt_tokens',
    right_on='context_tokens',
    how='inner',
)
display(merged)

plt.figure(figsize=(9, 4))
plt.plot(
    merged['prompt_tokens'],
    merged['prefill_ms'],
    marker='o',
    label='Prefill: full prompt forward',
)
plt.plot(
    merged['prompt_tokens'],
    merged['decode_ms'],
    marker='o',
    label='Decode: one new token forward',
)
plt.xlabel('Prompt / cached context length')
plt.ylabel('GPU execution time (ms)')
plt.title('NVIDIA T4 — Prefill versus decode')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Experiment 4 — Output length and full generation time

We now hold the prompt constant and vary the requested output length. `min_new_tokens` and `max_new_tokens` together reduce early-EOS variation.

In [ ]:
# Prepare one realistic instruction prompt for end-to-end generation timing.
messages = [{
    'role': 'user',
    'content': 'Explain why prefill and decode have different GPU behavior.',
}]
encoded = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors='pt',
).to(device)

@torch.inference_mode()
def generation_benchmark(new_tokens, repeats=5):
    # The generated length is constrained so output-length comparisons are useful.
    def run():
        return model.generate(
            **encoded,
            min_new_tokens=new_tokens,
            max_new_tokens=new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Warm up before wall-clock timing.
    _ = run()
    torch.cuda.synchronize()
    times = []
    generated_count = 0
    for _ in range(repeats):
        torch.cuda.synchronize()
        started = time.perf_counter()
        output = run()
        torch.cuda.synchronize()
        times.append((time.perf_counter() - started) * 1000)
        generated_count = output.shape[1] - encoded['input_ids'].shape[1]
        del output
    return statistics.median(times), int(generated_count)

generation_rows = []
for new_tokens in [8, 16, 32, 64]:
    milliseconds, actual = generation_benchmark(new_tokens)
    generation_rows.append({
        'requested_tokens': new_tokens,
        'generated_tokens': actual,
        'total_generation_ms': milliseconds,
        'output_tokens_per_sec': actual / (milliseconds / 1000),
    })

generation_df = pd.DataFrame(generation_rows)
display(generation_df)

The total time includes prefill and all decode iterations. A useful approximation is:

$$T_{\text{total}}\approx T_{\text{prefill}}+(G-1)\overline{T}_{\text{decode}}+T_{\text{other}}$$

Short outputs may be dominated by prefill and fixed overhead; longer outputs expose incremental decode more clearly.